# Cache Magi character embeddings

Use the **Godfather** kernel and run top to bottom. Read every baseline `magi_raw.json`, use its exact character boxes, and compute frozen MagiV2 crop embeddings **on GPU**. Save `character_embeddings.json` beside each raw cache.

No detection, OCR, GT matching, character cleanup, identity assignment, or training runs here. Original pages and character indices stay intact. All appearances are embedded, including those without dialogue.


## 1. Imports and settings

<details>
<summary>Details</summary>

CUDA is required. The model loads once, on the first embedding cache miss. `SEQUENCE_LIMIT=None` processes all cached sequences. Lower `EMBED_BATCH_SIZE` if GPU memory is limited.

</details>


In [1]:
import torch
import gc
import hashlib
import json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from transformers import AutoModel

assert torch.cuda.is_available(), 'Select Godfather with CUDA-enabled PyTorch.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(DEVICE))

MAGI_ROOT = Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi')
MODEL_PATH = Path('C:/MODELS/magiv2')
SPLIT = 'development'
IMAGE_ROOT = MAGI_ROOT / 'TaskData' / SPLIT / 'images'
CACHE_ROOT = MAGI_ROOT / 'paddleOCR-Magi' / 'cache_baseline' / SPLIT
MODEL_REVISION = 'local-magiv2-1'
EMBEDDING_DIM = 768
EMBED_BATCH_SIZE = 64
EMBED_VERSION = 1
FORCE_EMBEDDINGS = False
SEQUENCE_LIMIT = None

assert MODEL_PATH.is_dir(), f'Missing model: {MODEL_PATH}'
assert EMBED_BATCH_SIZE > 0
sequence_folders = sorted(path.parent for path in CACHE_ROOT.glob('*/magi_raw.json'))
if SEQUENCE_LIMIT is not None:
    assert SEQUENCE_LIMIT > 0
    sequence_folders = sequence_folders[:SEQUENCE_LIMIT]
assert sequence_folders, 'Run magiBasePredCache.ipynb first: no raw caches found.'
print('Cached sequences to process:', len(sequence_folders))


GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Cached sequences to process: 80


## 2. Cache helpers and model fingerprint

<details>
<summary>Details</summary>

Use the same cache wrapper and model fingerprint as the baseline notebook. Checksums detect changed data. The embedding signature includes raw prediction provenance, original boxes, ordered page hashes, model metadata, preprocessing, and package versions. Valid embedding caches are reused.

</details>


In [2]:
def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def digest(data):
    text = json.dumps(json_ready(data), sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def file_hash(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(data), indent=2, ensure_ascii=True,
                                    allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def cached_stage(folder, name, settings, compute, force=False):
    signature = digest(settings)
    path = folder / f'{name}.json'
    if path.exists() and not force:
        try:
            saved = json.loads(path.read_text(encoding='utf-8'))
            if saved['signature'] == signature and digest(saved['data']) == saved['data_hash']:
                return saved, 'cache'
        except (ValueError, KeyError, OSError):
            pass
    data = json_ready(compute())
    saved = dict(signature=signature, data_hash=digest(data),
                 created_at=datetime.now(timezone.utc).isoformat(), settings=settings, data=data)
    write_json(path, saved)
    return saved, 'computed'


model_files = []
for path in sorted(MODEL_PATH.rglob('*')):
    if path.is_file() and path.suffix in {'.json', '.py', '.safetensors', '.bin', '.pt', '.pth'}:
        stat = path.stat()
        model_files.append(dict(name=str(path.relative_to(MODEL_PATH)), size=stat.st_size,
            mtime_ns=stat.st_mtime_ns, sha256=file_hash(path) if path.suffix in {'.json', '.py'} else None))
MODEL_INFO = dict(path=str(MODEL_PATH), revision=MODEL_REVISION, files=model_files)
PACKAGES = {name: version(name) for name in ['torch', 'torchvision', 'transformers', 'numpy', 'pillow']}


## 3. Validate raw inputs and alignment

<details>
<summary>Details</summary>

Take page filenames from the raw cache metadata, never from a fresh directory sort. Confirm source image hashes and model fingerprint still match the baseline. A mismatch is reported as an error; regenerate the baseline for changed inputs. No GT labels are read.

</details>


In [3]:
def read_raw_sequence(folder):
    cache = json.loads((folder / 'magi_raw.json').read_text(encoding='utf-8'))
    assert digest(cache['data']) == cache['data_hash'], 'Raw cache checksum mismatch'
    assert digest(cache['settings']) == cache['signature'], 'Raw signature mismatch'
    settings = cache['settings']
    assert settings['sequence_id'] == folder.name and settings['split'] == SPLIT
    assert settings['model'] == MODEL_INFO, 'Model differs from baseline; regenerate raw cache'
    assert settings['preprocessing'] == 'grayscale_then_RGB', 'Unexpected baseline preprocessing'
    metadata = settings['pages']
    assert len(metadata) == len(cache['data']), 'Page metadata/results mismatch'
    assert len({page['name'] for page in metadata}) == len(metadata), 'Duplicate page filenames'
    page_files = []
    for page in metadata:
        assert Path(page['name']).name == page['name'], 'Expected a page filename'
        path = IMAGE_ROOT / folder.name / page['name']
        assert path.is_file(), f'Missing page: {path}'
        assert file_hash(path) == page['sha256'], f'Changed source page: {path.name}'
        page_files.append(path)
    for page in cache['data']:
        for box in page['characters']:
            values = np.asarray(box, dtype=float)
            assert values.shape == (4,) and np.isfinite(values).all(), 'Invalid character box'
            assert values[2] > values[0] and values[3] > values[1], 'Degenerate character box'
    return cache, page_files


def validate_embedding_data(data, raw_pages, page_metadata):
    assert data['embedding_dim'] == EMBEDDING_DIM and data['normalization'] == 'none'
    assert data['dtype'] == 'float32'
    assert len(data['pages']) == len(raw_pages) == len(page_metadata)
    for page_index, (saved, raw, metadata) in enumerate(zip(data['pages'], raw_pages, page_metadata)):
        assert saved['page_index'] == page_index and saved['page_file'] == metadata['name']
        assert len(saved['characters']) == len(raw['characters']), 'Character/embedding count mismatch'
        for character_index, (record, box) in enumerate(zip(saved['characters'], raw['characters'])):
            assert record['character_index'] == character_index
            assert record['character_id'] == f'PG{page_index+1}:C{character_index+1}'
            assert record['box'] == box, 'Character box/order mismatch'
            vector = np.asarray(record['embedding'], dtype=np.float32)
            assert vector.shape == (EMBEDDING_DIM,) and np.isfinite(vector).all(), 'Invalid embedding'


## 4. Frozen GPU encoder and ordered records

<details>
<summary>Details</summary>

Call `predict_crop_embeddings` for one page at a time, supplying the raw boxes in their original order. Magi returns one row per input box; package that row with the same box and index. Empty pages remain in place. Save raw, unnormalized 768-dimensional CLS vectors as float32 values for later projection training. L2 normalization for cosine baselines can happen when consuming these embeddings.

</details>


In [4]:
magi_model = None


def get_magi():
    global magi_model
    if magi_model is None:
        magi_model = AutoModel.from_pretrained(str(MODEL_PATH), trust_remote_code=True,
            local_files_only=True).to(DEVICE).eval()
        magi_model.requires_grad_(False)
        assert not magi_model.config.disable_crop_embeddings, 'Crop encoder is disabled'
        assert magi_model.config.crop_embedding_model_config.hidden_size == EMBEDDING_DIM
        assert next(magi_model.crop_embedding_model.parameters()).is_cuda, 'Crop encoder must be on GPU'
    return magi_model


def embedding_records(page_index, page_file, boxes, vectors):
    assert len(vectors) == len(boxes), 'Magi returned a different crop count'
    assert vectors.ndim == 2 and vectors.shape[1] == EMBEDDING_DIM
    assert torch.isfinite(vectors).all(), 'Nonfinite crop embeddings'
    vectors = vectors.detach().to(device='cpu', dtype=torch.float32).tolist()
    return dict(page_index=page_index, page_file=page_file, characters=[
        dict(character_index=i, character_id=f'PG{page_index+1}:C{i+1}',
             box=box, embedding=vectors[i]) for i, box in enumerate(boxes)])


def compute_embeddings(raw_pages, page_files):
    pages = []
    for page_index, (raw, path) in enumerate(zip(raw_pages, page_files)):
        boxes = raw['characters']
        if not boxes:
            pages.append(dict(page_index=page_index, page_file=path.name, characters=[]))
            continue
        with Image.open(path) as image:
            page_image = np.array(image.convert('L').convert('RGB'))
        with torch.inference_mode():
            vectors = get_magi().predict_crop_embeddings(
                [page_image], [boxes], mask_ratio=0.0, batch_size=EMBED_BATCH_SIZE)[0]
        assert vectors.is_cuda, 'Expected GPU crop embeddings'
        pages.append(embedding_records(page_index, path.name, boxes, vectors))
        del vectors, page_image
    return dict(embedding_dim=EMBEDDING_DIM, dtype='float32', normalization='none', pages=pages)


## 5. Cache all sequence embeddings

<details>
<summary>Details</summary>

Run this cell for the GPU batch. Failed sequences are logged and the batch continues. Completed caches survive an interruption; rerun to resume. This writes only new embedding files and an embedding batch report, leaving raw predictions, GT matches, and their manifests unchanged.

</details>


In [6]:
embedding_report = []
try:
    for number, folder in enumerate(sequence_folders, start=1):
        row = dict(sequence_id=folder.name, status='running', cache_status=None, error=None)
        print(f'[{number}/{len(sequence_folders)}] {folder.name}', flush=True)
        try:
            raw_cache, page_files = read_raw_sequence(folder)
            raw_pages = raw_cache['data']
            settings = dict(algorithm_version=EMBED_VERSION, sequence_id=folder.name,
                magi_signature=raw_cache['signature'], magi_data_hash=raw_cache['data_hash'],
                pages=raw_cache['settings']['pages'], boxes_sha256=digest([p['characters'] for p in raw_pages]),
                model=MODEL_INFO, packages=PACKAGES, preprocessing='grayscale_then_RGB',
                embedding_dim=EMBEDDING_DIM, dtype='float32', normalization='none',
                mask_ratio=0.0, batch_size=EMBED_BATCH_SIZE)
            saved, row['cache_status'] = cached_stage(folder, 'character_embeddings', settings,
                lambda: compute_embeddings(raw_pages, page_files), force=FORCE_EMBEDDINGS)
            validate_embedding_data(saved['data'], raw_pages, settings['pages'])
            row['pages'] = len(raw_pages)
            row['characters'] = sum(len(p['characters']) for p in raw_pages)
            row['status'] = 'ok'
            print(f"  {row['cache_status']} | pages={row['pages']} | characters={row['characters']}", flush=True)
        except Exception as error:
            row['status'] = 'error'
            row['error'] = f'{type(error).__name__}: {error}'
            print('  ERROR:', row['error'], flush=True)
            # Reload after failures (e.g. an interrupted encoder call with changed mask state).
            magi_model = None
        embedding_report.append(row)
        write_json(CACHE_ROOT / 'embedding_batch_report.json', dict(split=SPLIT,
            updated_at=datetime.now(timezone.utc).isoformat(), sequences=embedding_report))
        gc.collect()
        torch.cuda.empty_cache()
finally:
    magi_model = None
    gc.collect()
    torch.cuda.empty_cache()

display(pd.DataFrame(embedding_report))
successful = [row for row in embedding_report if row['status'] == 'ok']
print('Successful sequences:', len(successful), '/', len(embedding_report))
print('Character embeddings cached:', sum(row['characters'] for row in successful))


[1/80] seq_032cbb4dc8686963
  cache | pages=3 | characters=18
[2/80] seq_06ec0acde37dffa4
  cache | pages=3 | characters=18
[3/80] seq_07838fb26cd1be54
  cache | pages=3 | characters=16
[4/80] seq_098e260f231aa789
  cache | pages=3 | characters=29
[5/80] seq_0dc2e3d874c804fc
  cache | pages=3 | characters=31
[6/80] seq_0ef2ec294597f572
  cache | pages=3 | characters=23
[7/80] seq_1c1d4f5e71a2f554
  cache | pages=3 | characters=24
[8/80] seq_2022b8c0e1f2d3c0
  cache | pages=3 | characters=25
[9/80] seq_2032620aa4e4ac7f
  cache | pages=3 | characters=34
[10/80] seq_210721b0d576149f
  cache | pages=3 | characters=10
[11/80] seq_27e88136c1812f85
  cache | pages=3 | characters=17
[12/80] seq_2e4b837ead6b88ae
  cache | pages=3 | characters=13
[13/80] seq_2fae297a18af2a57
  cache | pages=3 | characters=22
[14/80] seq_41d68dfc426cc19e
  cache | pages=3 | characters=21
[15/80] seq_44b9a316b2919cfe
  cache | pages=3 | characters=19
[16/80] seq_45c9871f52e99417
  cache | pages=3 | characters=25
[

,sequence_id,status,cache_status,error,pages,characters
0,seq_032cbb4dc8686963,ok,cache,None,3,18
1,seq_06ec0acde37dffa4,ok,cache,None,3,18
2,seq_07838fb26cd1be54,ok,cache,None,3,16
3,seq_098e260f231aa789,ok,cache,None,3,29
4,seq_0dc2e3d874c804fc,ok,cache,None,3,31
...,...,...,...,...,...,...
75,seq_f9a4e67ae3c018f6,ok,computed,None,3,22
76,seq_fa2eee6269a5dad3,ok,computed,None,3,24
77,seq_fa5b05d0f22bff15,ok,computed,None,3,19
78,seq_fb6d15ad78b7191c,ok,computed,None,3,23


Successful sequences: 80 / 80
Character embeddings cached: 1973


## 6. Inspect the mapping

<details>
<summary>Details</summary>

Each appearance stores its exact original box next to its embedding. The page list and character list preserve raw Magi ordering. `page_index` and `character_index` are zero-based; display IDs are one-based. No GT identity is attached.

</details>


In [7]:
if successful:
    INSPECT_SEQUENCE = successful[0]['sequence_id']
    saved = json.loads((CACHE_ROOT / INSPECT_SEQUENCE / 'character_embeddings.json').read_text(encoding='utf-8'))
    rows = [dict(page_index=p['page_index'], page_file=p['page_file'],
                 character_index=c['character_index'], character_id=c['character_id'],
                 box=c['box'], embedding_length=len(c['embedding']))
            for p in saved['data']['pages'] for c in p['characters']]
    display(pd.DataFrame(rows))
else:
    print('No successful embedding caches to inspect.')

# Later, for appearance PG1:C1:
# vector = saved['data']['pages'][0]['characters'][0]['embedding']
# box = saved['data']['pages'][0]['characters'][0]['box']


,page_index,page_file,character_index,character_id,box,embedding_length
0,0,01.png,0,PG1:C1,"[377.9719543457031, 886.2671508789062, 508.375...",768
1,0,01.png,1,PG1:C2,"[265.11004638671875, 422.6414794921875, 623.77...",768
2,0,01.png,2,PG1:C3,"[607.0160522460938, 852.071044921875, 728.1933...",768
3,0,01.png,3,PG1:C4,"[364.93072509765625, -2.548191785812378, 786.4...",768
4,0,01.png,4,PG1:C5,"[84.12738037109375, 808.1517333984375, 332.909...",768
5,1,02.png,0,PG2:C1,"[579.6659545898438, 935.4791259765625, 711.765...",768
6,1,02.png,1,PG2:C2,"[39.24248504638672, 1.2456532716751099, 220.10...",768
7,1,02.png,2,PG2:C3,"[171.4403533935547, 857.493896484375, 397.0146...",768
8,1,02.png,3,PG2:C4,"[401.578125, 0.9899213910102844, 834.791442871...",768
9,1,02.png,4,PG2:C5,"[434.6626892089844, 859.4111328125, 583.763488...",768


## Saved format

`cache_baseline/development/<sequence_id>/character_embeddings.json`:

```text
signature / data_hash / created_at / settings
data
  embedding_dim: 768
  dtype: float32
  normalization: none
  pages (original page order)
    page_index, page_file
    characters (original character-box order)
      character_index, character_id, box, embedding (768 values)
```

The embedding belongs to the `box` stored in the same character record. Indices refer to raw detections, including duplicates; they do not refer to pipeline1's cleaned characters. `settings` binds the file to the raw cache signature/checksum and source image hashes. Later training-data generation should verify these references before combining GT matches and embeddings.
